In [0]:
%pip install faiss-cpu sentence-transformers transformers torch --quiet
dbutils.library.restartPython()

In [0]:
%pip install -q \
  "datasets==2.19.0" \
  "huggingface_hub>=0.24.0" \
  "fsspec==2024.2.0"

dbutils.library.restartPython()

In [0]:
import os, base64, json, pickle
import numpy as np
import faiss
from transformers import AutoTokenizer, AutoModel
import torch
from datasets import load_dataset
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
hf_token_b64 = w.secrets.get_secret("vaidya-lipi", "hf_token").value
hf_token = base64.b64decode(hf_token_b64).decode("utf-8")

print("Loading Parrotlet-e model...")
model_name = "ekacare/parrotlet-e"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModel.from_pretrained(model_name, token=hf_token)
model.eval()
print("Model loaded")

def embed_texts(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                           max_length=128, return_tensors="pt")
        with torch.no_grad():
            output = model(**encoded)
            # Mean pooling
            attention_mask = encoded["attention_mask"]
            token_embeddings = output.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).float()
            embeddings = (token_embeddings * input_mask_expanded).sum(1)
            embeddings = embeddings / input_mask_expanded.sum(1).clamp(min=1e-9)
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
            all_embeddings.append(embeddings.numpy())
        if i % 500 == 0:
            print(f"  Embedded {i}/{len(texts)}")
    return np.vstack(all_embeddings)

print("Loading SNOMED English corpus (this is the search target)...")
corpus = load_dataset("ekacare/Eka-IndicMTEB", "corpus", split="test", token=hf_token)
texts = [row["text"] for row in corpus]          # English SNOMED terms: "ankle pain", "chest discomfort"
concept_ids = [row["concept_id"] for row in corpus]
# corpus is all English — no language field needed
print(f"SNOMED corpus size: {len(texts)} English concepts")
# At runtime: embed "टखने में दर्द" → Parrotlet-e maps it to same space → finds "ankle pain" (SNOMED 247373008)

print("Building embeddings...")
embeddings = embed_texts(texts, batch_size=64)
print(f"Embeddings shape: {embeddings.shape}")

# Build FAISS index (cosine similarity via normalized vectors + inner product)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner product = cosine for L2-normalized vectors
index.add(embeddings)
print(f"FAISS index built: {index.ntotal} vectors")

# Save locally first
os.makedirs("/tmp/parrotlet_index", exist_ok=True)
faiss.write_index(index, "/tmp/parrotlet_index/index.faiss")

metadata = [{"term": t, "concept_id": c}
            for t, c in zip(texts, concept_ids)]
with open("/tmp/parrotlet_index/metadata.json", "w") as f:
    json.dump(metadata, f)
print("Saved to /tmp")

In [0]:
# Upload to UC Volume so the app can access it
volume_path = "/Volumes/workspace/vaidya/models_and_indexes/parrotlet_index"
dbutils.fs.mkdirs(volume_path)

# Copy files to volume
dbutils.fs.cp("file:/tmp/parrotlet_index/index.faiss",
              f"{volume_path}/index.faiss")
dbutils.fs.cp("file:/tmp/parrotlet_index/metadata.json",
              f"{volume_path}/metadata.json")

print(f"Index uploaded to {volume_path}")
print(f"Files: {dbutils.fs.ls(volume_path)}")